# Automatic Music Generation (AMG)

The paradigm in Automatic Music Generation is that: Music Representation + Generation Model + Control = Pleasant and Useful Music

Human perceives music in 2 ways:
1. Listening = Audio domain (MP3, WAV, etc.)
2. Reading = Symbolic domain (Scores, MIDI Files, etc.)

<div>
<img src="../images/auto-music-gen/image-1.png" width="750"/>
</div>


<div>
<img src="../images/auto-music-gen/image-2.png" width="350"/>
</div>


With piano roll, we can express symbolic music as a binary valued matrix.

with dimensions, H * W * c.

H = the pitch range of the notes
W = time
C = Different instruments or channels

Symbolic music formats like MIDI is not directly understandable or computable by sequence models such as transformer. Therefore to leverage sequence models for AMG, we need to tokenize the symbolic music data uh transforming it into compatible encodings. Unfortunately when it comes to AMG models, MIDI and uh piano roll formats fall short in terms of efficiency.

## Symbolic Music Representation

**Traditional Formats -- with its limitation**
* MIDI:
    * Cannot be directly understood or computed by sequence models (like Transformers).
    * Lacks explicit bar, beat, and sub-beat information.
    * Inefficient for capturing rhythmic structures.
* Piano Roll:
    * Sparse representation.
    * Computationally inefficient for sequential AI models.

**Event Token-Based Music Encodings**
Event Token-Based Music Encodings transform MIDI files into sequences of event tokens similar to natural language processing. Tokens represent pitch, duration, positions, and other musical information.

1. REMI Encoding
* Converts MIDI events into sequential tokens.
* Adds bar tokens to mark the start of a bar.
* Adds position tokens to indicate specific locations within a bar.
* Purpose: Creates a metrical grid for beat- and bar-aware modeling.
* Example: A note’s pitch, duration, velocity, and its position in a bar are all represented by separate tokens.

2. CP / CB Encoding
* Condenses REMI tokens into super tokens for efficiency.
* Two types of super tokens:
    1. Musical tokens → encode notes and their attributes.
    2. Metrical tokens → encode beat/bar information.
* Reduces token sequence length and computational cost.
* Maintains rhythmic and musical structure while being more compact than REMI.

3. Octuple Encoding
* Unifies CP’s musical and metrical tokens into one type of token.
* Each token (octuple) contains eight key elements:
    1. Time signature
    2. Tempo
    3. Bar information
    4. Position in bar
    5. Instrument
    6. Pitch
    7. Duration
    8. Velocity
* Benefits:
    * Shorter input sequences.
    * Dense gradients → more efficient training.
    * Encapsulates all essential musical and metrical info in one token.

Encoding and the token size comparison approximation
1. REMI (let's say we have 15K Token number)
2. CP (we'd have 6.9K Token number)
3. Octuple (we'd have 3.6K Token number)

Less tokens means shorter inputs and denser gradient matrics which enhances the efficiency and effectiveness of model training. So the Octuple encoding is the most commonly used.

**Other Encodings**
* Textual: MusicXML, LilyPond, ABC notation.
    * Pure text; can work with large language models like ChatGPT.
    * Not widely used in AMG due to long sequences and inefficiency.

* Graph-Based:
    * Notes as nodes; relationships (timing, bars, track transitions) as edges.
    * Helps models recognize repetitive structures (e.g., chord substitutions, approximated variations).
    * Limitation: low compatibility with sequential models like Transformers.



There are also other encodings used in AMG. For example, three pure textual
encodings uh called one is music XML, the other is
lily pond and also ABC notations. But they are not so widely used in MG
because they are lengthy and they are uh
how to say too long to uh be loaded in a sequential model. But these encodings
are potential to be used in interactions with large language model like chat GPT
because these encodings are pure texts.
And symbolic music can also be encoded using graphs where notes are represented
as nodes and the relationships such as timing uh mayors, bars and track
transitions are represented as ages.
Experiments show that graph encoding helps models recognize the repetitive
structures in music because repeating structures that are not identical but
highly similar such as music transitions, chord substitution and approximated
variations uh in bars tended to have more consistent uh topological patterns
under graph encoding. However, the problem of graph encodings
is its limited compatibility with sequential models like transformers.
So in the above mentioned encodings, the same uh music note will always be
converted as the same token. But maybe that's not enough for AMG
models. So let's um do an experiment to
show why it is not enough and to show the significance of music embedding.
Let's take a look at the following example. There are three nodes and the middle note is so.
So uh in all the uh encodings it will be converted to the same token.
But is the not source meaning the same when it is surrounded by the notes here
or
does it has the same meaning when it is played in the intro part of a music or
in the chorus part of a music? And does its meaning remains consistent when I
say that this soul comes from a jazz fast tempo song or it comes from a
Russian classical compositions in 19th centuries?
The answer is no. The meaning of the same note so various among different
scenarios. This is why we cannot always use the same token to represent uh same nodes in
different u conditions in different contexts
and it is crucial to model the contextual informations of nodes. This brings us to the concept of embedding.
I will introduce several widely used embeddings in AMG in audio domain wave
to vake 2.0 uh the name of this embedding
it is a sales supervised embedding learning model. So the key idea is to
mask some audio frames and try to predict the masked positions back based
on its uh surrounding frames. The predicted result should be as similar
and the ground truth before mask while other positions predicted result should
be greatly different from the musk position. And this optimization idea is
called contrastive loss. And two similar works on symbolic music
embedding learning are MIDI bird and music bird. The training procedure is
very similar to wave to wake you know randomly masking some u tokens here and
try to reconstruct them from contextual tokens. The difference is just uh these
two embeddings use cross entropy loss instead of contrastive loss because symbolic music is discrete.
So after uh masking the tokens uh the models computation and reconstructions
the hidden states here are extracted as embeddings.
The other one uh is called Mulan embedding. It involves multiple
modalities and it is proposed to link music and texts. Mulan embedding model
trained to project music's audio embedding with its paired text description uh text embedding into one
shared hyperspace and optimize with the contrastive loss so that paired audio
clips and uh texts have similar embedding values.
After training text and audio features are decoupled. The audio embeddings now
convey their associate text embeddings while the text embeddings also
encapsulate their paired music features.
Okay, now let's dive into the second part of the uh diagram. Music generation
models. Researchers have long been attempting to
push in the span boundaries of music generation. Meaning they try to accommodate longer music uh accommodate
longer music inputs and computations uh longer notes in the generation. With
the continuous increase in computing power and the emergence of powerful models, AMG has transitioned from
looking at only one state to the whole piece.
So why do researchers aspire to accommodate longer music inputs? To
answer this question, we can start with another experiment. So the task is simple. We can do this
all together. listen to the initial segment of a music piece and try to guess the subsequent
notes. Namely, you will hear uh the previous notes and you should guess the
pitch and duration notes indicated by the uh arrows symbols here.
Okay, there are several runs. So, first let's proceed with the initial segments.
[Music] Okay, let's uh have a listen again.
So, what should be the next note? The original samples uh so here. So it
should be [Music] but for me I think uh
or also makes sense. So here are multiple
choices. Okay. Now next uh let's proceed with the
second segment. [Music]
Okay, this time what is the next note? This time it is easier, right? It can be
simply deduced based on the repetition rule. And last, let's proceed with the final
segment. [Music]
Okay. So, what's the next note? This time I guess everyone should hear a
strong voice saying put a to here to complete the song.
So, music is not random combination of notes. There should be rules of composition in music and based on what
we have, we can somehow predict what music could proceed next.
So the technique we all used in this experiment I believe using pass note to
anticipate the next note is known as auto reggressive.
Most music generation models primarily utilize this approach to consistently
predict and generate the next note thereby assembling them into a complete
music composition. Through these three instance of prediction, we can deduce that longer
proceding segments generally result in a higher probability of accurately
predicting the subsequent note. This insight highlights the significance
of music generation models understanding a wide spectrum of note dependencies.
Achieving this long-term dependencies allows the generative output to closely
resemble the original causations crafted by human hands.
Modeling preceding node sequences also presents a trade-off.
Integrating longer music sequences into model leads to improved prediction accuracy, but it also requires increased
computing power and more efficient representations. And now let's explore the steps that
music generation model have taken to broaden their perception scope of music.
In my opinion, I will position the starting point of AMG of auto
reggressive AMG with MOAT key. We can listen to this piano piece by Mozat.
First [Music]
Does anyone know the name of this Mosat piano composition?
Certainly no one knows because this piece of music is not a complete
complete creation of Mozat but rather a randomly generated one.
If you're interested in Mozart game, scan this QR code and try to create your
own Mozart piece. The method of generating music in the
style of Mozart is called Mozad. As I mentioned, I consider it as the starting
point of auto reggressive AMG. So back to 18th century, music lovers
used dice to randomly generate music from preomposed options. Music pieces by
Mozad were first segmented by bars and numbered.
Then the players keep toing two dice adding the numbers and pick up the
corresponding music bars to concatenate. Then uh repeat this process again and
again. In this way the music can go forever. The moaf game is auto reggressive
because as definition an auto reggressive model is using past values
to predict it current value. Especially in mo game the past value make zero
impacts on the prediction of next nodes. Thus we can say its respective field is
zero. From a computer scientist view, moat
game can also be regarded as an evenly distributed markoff process. There is a
very basic mark of chain model uh mechanism behind moat game. As shown in
the left figure in this marov process of moat games all nodes denote a different
music segment of Mozart also we called as states in mark of chains h here
denotes the current music segment and t1 t2 and blah blah dis all the t's are
other segments so the probabilities of transitions from any state to another
state are all the same. Each segment can be selected with the
probability of one over the total um number of segments.
Markoff model is later widely used in AMG. In the previous Mosat game, the
states in the mo marov model are pre-cut musical segments composed by mozzat
based on this idea of markov models state transition. Many AMG studies
proposed a bunch of markov models to generate melodes
uh core progression and also note durations. Uh for example, we can use three
separate markoff models to predict the pitch of next melody note to predict the
a complement uh chord and also to predict the note duration
as shown in the diagram. If uh current state now
uh is a me fourth note with um F chord
accompaniment, right? And then
next note could be a to pitch eighth note accompanied with a chord C.
But there can be other you know possibilities. The probability used in the transition matrices uh of these
markup models for note transitions, accompaniment core transitions and music
durations can be uniformly distributed as seen in the Mozart game and they can
be also derived from mimicking an existing music data set uh
mathematically forming compositional rules or they can come from just random
generation. This is what we generated based on this
uh uh nose transition. [Music]
The limitation of using markup models for music generation are apparent in a mark of random process. The next state
probability distribution is sorely dependent on the current state. Namely,
its receptive field is only one. This limits mark of based AMG's ability to
incorporate a significant amount of previous information.
There is a demo to show this problem. [Music]
The generated notes lack continuity not forming a melody.
You may ask why not consider more previous states.
We can do that uh but the cost is exponentially increasing the number of
states and also expanding the size of transition matrix.
Then a multi-layer perception is proposed to address the issue of
exploding states in giving for example a window of four
musical nodes as input. It predicts the probability of next window of the
subsequent four nodes based on the neuronet network's parameters rather than relying on a state transition table
like in micro model. So its receptive field is longer. It is
the length of the input window. But unfortunately within each input window,
the model also ignores the progress nodes.
It also mean that MLP can only process inputs of the same length of the window
length. Right? And additionally, its complexity still grows geometrically
along a window size, making it unsuitable for handling long sequences
of music due to the computational power constraints back to that time.
In contrast to MLPS, RNN uh based AMG addresses issues related to
exploding parameters, fixed length input, and the lack of considerations for context.
With RNN, we process inputs one note at a time while incorporating the
historical computation results to influence the generation of the current
music notes. Because in uh music inference
the looping output is iteratively input to the next step of RN. As the outputs
continue to loop the network essentially has a record of all previous received
information within the hidden states.
This is a demo uh generated by a ROM based music generator.
[Music]
So the problem of RNN and STL u RN LSTM based AMGs are uh number one slow
computation for long sequence because they deal with one note at each step
and although RN's uh receptive field can be infinite long in theory it suffers
from vanishing gradient issue meaning the far previous notes make very little
impact. uh to the update of the network parameters. Thus, the effective
receptive field is only like dozens of nodes.
With the rapid increase in computing power, structures resembling MLP have
once again captured people's attention. However, this time with a fresh approach
in the form of music transformer. It has revolutionized the landscape of
automatic music generation. The core of uh music transformer is of
course self attention and advantages of self attentions are that each computing
unit can capture information from the entire sequence which allows longer
dependence and the output B1 to B4 here can be
computed in parallel speeding up the training process. Also compared with MLP
its attention is is dynamically decided according to the input values making it
more powerful in sequential modeling.
Specifically the music transformer uh takes MIDI file as input. The series of
MIDI control codes in the MIDI file are encoded uh using bramy encoding as we
have introduced which are then converted into uh vectors through an embedding
layer to facilitate uh computation and musical contextual modeling.
These embedding vectors pass through a stack of self- tension blocks of the transformer generating an equal number
of vectors uh with updated values through each uh block. And during this
process uh each vector refers to informations from other positions via
their via um the attention mechanism. And finally, a linear layer followed by
a soft max function is used to predict the next music token which is appended
to um the input sequence. This auto reggressive generation process is
repeated until uh the whole music is finished.
So by visualize uh by visualizing the process of music transformer it becomes
apparent that uh in music transformer when predicting the next note it's the
theoretical reference scope the receptive field can encompass all the
previous encountered uh notes. As a result, music transformer has made
significant advancements uh in terms of the structural
understanding and long-term dependencies in AMG. Here we have a video visualization of
music transformers dependency in uh its music generating process.
Heat. Heat. [Music]
So we can see the receptive field of music transformer is quite long can be
hundreds of notes. Even the first few nodes can affect the note generated in
the end.
Okay. So now we have a very interesting survey to do. If you want to be a Pokemon trainer, we
can now do this together. So here are six music generation demos from some AMG
models. I'd like to set the model's name up for suspense. So, I use six different
Pokemon EVs to represent them. So, I will play the six pieces one by one, and
you could scan the QR code now to vote based on uh which music demo is the best
overall. uh which one is most structural, most uh humanlike and which one is most stable
and which one matches the description of relaxing jazz the best.
Let's reveal the winners after. So now I will play the first one, the
electronic EB.
Okay. The second is a water E
[Music]
And then the fire even.
[Music]
And next is ice. [Music]
And next one is original E. [Music]
And the last one, the grass EV. [Music]
Okay, now you can scan the QR code and choose your Pokemon.
From MLP to music transformer, we use nom for
sitting notes to predict future notes. This is why we refer the models as
uh decoders. Once the input is established, the distribution of future
notes becomes certain. So their generation usually requires a short
music fragment fragments as leading primer.
To create entirely new music without relying on any preceding nodes, we will
need a generative model. Please note that generative models and
decoders are not mutually exclusive. You can think of a generating model as a
sampler. It randomly samples from a gshian noise distribution and this
sample when passed through any decoder mentioned earlier can produce an
entirely new music composition.
The first commonly used generative model is variational autoenccoder which has a
simple training principle. It compresses images or in our case
music into a latent space Z and then attempts to reconstruct the compressed
vector back into the original image or music. When we need to generate new
music, we simply sample a point from the latent space and decode it with a
decoder.
An AMG example using uh VAE variational encoder is jukebox.
Uh you don't need to worry about the three rows here. They are basically the same. They just try to compress and
reconstruct music in different resolution and see which resolution works best. So let's take at the root uh
on the top for example. It starts by using transformer to encode audio and
then using a VQVA to further compress it
into discrete codes. The so-called uh latent space Z. This compression capture
informations about melody, rhythm, compositional uh characteristics and
timbers of various instrument as well as the styles and voices of singers. Then
it use similar structure uh decoders to generate audios from the sampled um
codes. Let's listen to a demo.
Doo shark.
[Music]
You can say the demo is really like human singing, right? But sometimes meaningless lyrics can appear because I
guess it samples point of the non-existing songs from the latent space.
Okay, the second generative model is generative advisorial network GM which
consists of a generator and a discriminator. So um typically the generator and
discriminator can be any of the decoders we mentioned earlier to uh or they can
be other models. The principle uh behind GN is to train the generator to create
um new samples from scratch. Uh the re to generate really good samples that can
effectively confuse the discriminator to convince the discriminator that the
generation the generated result is from human beings. while simultaneously we
also train the discriminators to differentiate between generative and
real samples. So after the parallel training we got a powerful generators
and then we will use um the trained generator to create new music samples.
Um this paper is an example of using GN to generate music. So it's u core idea
is in the discriminators and the discriminator's task is simple
is to uh taking the spectrogram of the generated music of or maybe the real
music and then to analyze if the spectrogram uh shows some patterns some
repetit repetitive patterns that are usually seen in human compositions. Then
after trainings uh the generators are believed to you know be able to create
music with such kind of uh repetitive characteristics.
[Music]
Okay. I think you also have heard the name diffusion uh for image generation.
But in the context of music generation, one example of applying the concept of
our third generative model diffusion model to music generation is called
refusion. The idea behind refusion is quite straightforward. It generates an
image from a given description of the music that you want. However, this uh
generated music uh this generated image is not a regular image but a spectrogram
which corresponds to a clip of music. So here is a demo that I generated using
refusion uh with the prompt a song in Singapore style representing the spirit
of sound and music computing. [Music]
[Music]
I agree that it is Singapore style, but does it represent the spirit of sound
and music competing? I don't know. How do you think?
So, please note that refusion can have clips of original compositions in their
generated music which may cause copyright issue. And those who are interested in refusion
can scan this cure code to try to generate your own music.
As spectrogram is also image, let's take image generation to explain how
diffusion works. So diffusion model aims to progressively remove noises from a
randomly sample gshion noises and each state it removes some noises ultimately
resulting in high quality images. To be more specific, diffusion trains a
den noising model that takes both noise and a denoising state comp n as inputs
and produces the image after one den noising step. So when it comes to text
to image generation task the approach is straightforward. In addition to the
input noise and denoising state number, you also include an additional textual
description as a condition in the inputs like a cat in the snow.
The fusion model has proven to yield super real results compared to VAE and
G. Researchers speculate that this is uh because diffusion model effectively
blends the strength of both auto reggressive and non auto reggressive
models in each denoising state. it performs
less computational work than VAE because in VAE
the model wants to like reconstruct from noise to the original uh pictures in one
step but uh in diffusion it just you know the noise a little bit. So this
enhances the efficiency of this non-auto reggressive denoising process.
Furthermore, across all the noising steps, it also leverages the benefits of
a sequential auto reggressive model. So starting from noises under the
control of textual descriptions, the noising step by step. Finally, diffusion
models can both generate a video image of, for example, a photograph of an
astronaut riding in a horse or a spectrogram image of a piece of music.
Now, let's explore two state-of-the-art AMG models that can serve as strong
baselines if you're looking to delve into this field and develop your own AMG
models. The first one is uh the pop music transformer. It is based on the
original music transformer, but it has the following advantages. Number one, it
enhanced data processing efficiency by using Remy encoding and it uses
transformer XL. So, uh transformer XL is a fusion of
transformer and RM models to accommodate even longer music inputs. And number
three, it reuse the uh relative position embedding. It is not so uh important.
And finally, it integrates the beat tracking and the chord accompany uh
recognition into the encoding of the music. And this will result in a better
understanding of musicality. And let's just listen to a demo.
While there may be occasionally discrepancies between melody and chords, overall I think the model delivers a
high quality output. By considering these aspects, I believe
pop music transformer model offers a strong foundation for exploring AMG.
And if you can remember, it is the electronic EV in the survey. So how many
Pokemon trainers choose this one?
And in the field of audio music generation, the current state-of-the-art
open source approach is music LM. Music LM also supports uh text to music
generation, but it does not rely on, you know, generate a spectrogram image as an
intermediate medium. Instead, it directly utilize moonline embedding to
match input uh descriptions to the corresponding text audio embedding
which is then uh transformed into an OIC
or acoustic embedding. Sorry. uh when then the uh text audio embeddings
generated from the text description is transformed into an acoustic embedding
and then the acoustic embeddings can be decoded as music. So do not worry about
the architecture here. You can check it later anytime.
Music Loom produces high quality music compared to Refusion and offers a better
alignment uh with the input description. Moreover, it has better open-source
availability than Refusion. But I have found that it has also limitations.
Let's listen to the demos before I tell you what's the limitations.
[Music]
Okay. So the limitation of uh music OM is it requires a very specific and
detailed description of the music to be generated. This can make it difficult
for nonprofessionals to accurately convey the kind of music they want. For
example, the arcade uh game music, catchy guitar reef and symbol crash and
drum roll here are all kinds of professional terms and I have no idea
what they mean before. Okay, now uh we move on to the last part
of the diagram control.
Sometimes we want to create music with um precise emotions, styles or
distinctive essence of a specific composer. There are also situations
where we require music to complement photos or videos. Moreover, there are
instances when we aim to generate music that matches our uh provided
descriptions or personal reference. To accomplish this goals, it is
essential to integrate robust feature control into uh music generating models.
This feature control empowers us to direct the music generation process and
shape the predicted music notes in accordance with our intended requirements.
We will introduce several controlling methods for AMG in our last part of
parody. In early stage, researchers control AMG
with uh compositional rules in Markov models. As an example, we can
dynamically modify the transition matrix based on uh the current position
and the music key in the uh AMG. For example, when reaching the last note of
a piece, it is likely that we uh that the model should pick the key root note
to create a sense of completion. So we can adjust the transition matrix when
reaching the end by increasing the probability of the key root note.
For example, in the very first experiment we do together,
[Music] we all have a strong feeling that the
music will end with
right. We have a demo of such rule-based control in AMG. Uh the benefit of
rulebased control is a high stability of generating results across multiple
trials. However, it drawbacks lies in the lack of diversity and creativity.
Let's uh listen to the demo.
[Music]
and it is a fire a in the survey. So how many trainers pick this one?
In AMG controlling emotion is a common practice. Typically scientists use two
real numbers to represent the different emotions of a song. Uh their arousal and
balance. So a typical method for emotional control involves associating these two
real values with the input notes during training. And then when generating music
you can simply align it with two desired emotional values. The results are
however not so good because it is difficult for the generative model to
understand the meaning of these two values by you know just appending them to the music tokens.
And when the needs arises to generating music for visual content, for example,
uh picture, we can extract the visual features from
the image and input them along with the music notes into the decoder or generated model for joint training.
However, this uh forced pairing method um like mentioned above the uh
sentimental the emotional control and this uh like image control
uh do not yield very effective results and lack interpretability because this
control approach may seem somewhat forced and lacks meaningful insights
into the relationship between visual and musical modalities.
In contrast, this work we are discussing which is also uh centered on generating
music from videos offers a clear and insightful approach. It happens to be
one of my favorite uh model because you know in many crossmodel
tasks involving the transition from visuals to music conventional method
simply feed video and audio data into the generated model expecting it to
learn the connections between the two modalities. However, uh the authors of this work
made a pivotal observation. They recognize that both videos and music are
temporal art forms and their essential connection lies in the synchronization
of rhythmic changes. So they analyzed the timing of visual
scene changes, the intensity and also the speed of scene changes and then they
mapped these changes to beats, note dynamics and note densities of the
generated music. In essence, when the motion is fast, the notes should be
shorter but denser. When the motion is striking and intense, the music notes
should be louder and the note beats should also align with the timing of the
motion. I like this work because its idea is remarkably simple and intuitive yet it
yields impressive results and surprisingly no one had thought of it before. It gives a sense of
enlightenment as if um covering something hidden in the plain sight.
Here is a demo of this work. On the left is a
video and the generated music and on the right is the motion changes which
provide interpretability of its AMG. [Music]
[Applause] [Music]
Okay. So when it comes to generating music for lyrics, a critical consideration is singability.
Some mass accomplish the singability by utilizing existing song and lyrics data
to create both a course and fine grain alignment matrix between uh music and
lyrics. This matrix guides a model in generating a lined notes for each
syllable of input lyrics and also generating a whole sentence for a whole
music phrase ensuring that the music is easily singable. Let's listen to a demo.
Another
controlling method I want to introduce to you. This is how uh music LM achieves
controlling music based on textual description. The uh first it trains
mulan embedding model which connects text with its paired music. And then
giving a new description, it calculates the moola embedding which contains uh
both sematic and the musical information. It predicts the acoustic embedding right
from the moola embedding and decoded as a music that matches to the description.
The model here is simplified and more details are in their original paper.
Let me pose a question for everyone to think about given that Mulan already
incorporates matched textual and musical data, right? So why not just you know
input text during the inference stage to obtain moola embedding and then directly
decode it into music. Why do we first predict the acoustic
embedding and then decode it into music? I encourage you to think about this and
u if you're keen on finding the answer, please feel free to contact me.
This year the music generation system has made a groundbreaking process in
text to music tasks starting music industry. While the technical details
haven't been revealed, uh it can be speculated that uh uh the main uh
contribution is by the DIT architecture. DIT means diffusion with transformer.
So previously the diffusion model utilize a unit for denoising
essentially performing convolution and deconvolution on images. limiting it to
uh processing a single spectrogram at a time in music. In contrast, SUNO employs
a structure resembling transformer to handle audio spectrograms. It slices the
spectrograms into tokens and map them discretely, enhancing the continuity and
quality of generated uh audio segments.
Sunno's music generation skills are pretty good, almost as good as beginner songwriters can do. It can create long
pieces of music that fit well with lyrics and its UI is easy to use. Now,
let's check out the demo to see what it can do uh using some lyrics created by
Chess GPT. aiming for a style similar to uh Singapore's music style, happy uh and
with a clear voice. Here's uh the song it generated from the lyrics and prompts
[Music]
in the digital realm where dreams take flight.
is power ignites topics in media where animation
from lines of code emerges the dreams
AI proess in melodies its intopics of media where creativity
breeze bites and beats in perfect In the
age, new worlds bring unleashing potential.
Minds align in the fusion of our AI define
topics in the where limits retire. In the spark of creation, we rise higher.
We rise higher. AI's power in music's embrace
topics and media digital space where rhythms evolve and although
[Music]
It doesn't yet make music with complex structure than that humans do. You can't
really tweak or fine-tune music after it's made. And the song it makes don't
have the same performance skills as humanmade ones. Also, the songs it makes
can, you know, start to sound too similar and kind of lack a personal
touch, feeling too much like they're made by AI.
And recently the seed music platform has also expanded to support uh more
modality of controls including uh the text descriptions lyrics as you know and
it also supports uh conditioned on reference audio like humming and also
symbolic music like uh lead sheet. So the approach involves using different
encoders to generate prompts from these multimodel inputs which are then used to
sample from the music's latent space and synthesize audio.
For example, this demo generate a song using u the lyrics as a text uh prompts
and also it uses a MIDI lead sheet as the um symbolic music prompts and then
it will generate uh uh let's first listen to how the MIDI uh melody is
type. [Music]
So we input this uh MIDI prompt uh you know to inform the model that we want to
generate uh audio sound that using this melody but also with the lyrics we input
and then the model will give you something like Beneath the moon I can hear your heart
whisper secrets to me. Kiss me dreaming for nights and dreams beneath the moon.
Beneath the moon speed is one beneath the moon.
In your eyes I see forever lasting
time. We'll be together holding on to the sweet
ming dream.
Uh so there are some extended content uh for those interested you can explore
this topic further on your on your own. So this page uh basically tells you how
unit integrates prompt embeddings into each step of the downsampling and
upsampling process for control. It means how the descriptions u make difference
uh make impacts on the music generation process.
And similarly this is how uh the descriptions uh make impact
uh in the uh DIT architecture. You can also check it uh check it out if you're
interested. And another uh key technology that
enables transformer to generate audio music is uh the ability to compress
audios into tokens and restore it without significant losses. So there are
many milestones um methods like some string in codec and uh descriptives
methods and feel free to del into the specific technologies involved.
Although AI generated songs have become quite advanced they still seem to lack
something. Let's share my observations. So how the
kind of AI feelings uh where does the AI feelings come from? So I will use this
as a demonstration. I selected a set of lyrics and compared a human composed
version of music with an AI generated music.
Specifically, we cut and stitched uh segments from
both pieces at the beginning uh 30%, 50%
and the end along with their corresponding lyrics.
So, let's start by listening to uh both versions.
First, we will hear the human composed song. [Music]
Moonlight on where we first came below.
Moonlight [Music] on where we let each other go.
with you from spring to fall. That was where you were all that I could
know. Like me, like D,
like frog, like snow. That in life.
I beg you say
no. [Music]
Then we will listen to the generated music uh demo.
[Music] Moonlight on where we first came to
know. Moonlight on where we let each other go. With you from spring to fall.
That was when you were all that I could know.
Like mist, like dew, like frost, like snow. Hit
that in next life. I beg you say no.
[Music]
So in the human composed song each segment represents a clear motive uh
which is a music term and all four sections share a consistent style. More
importantly the motives are connected creating a sense of cohesion and a
similarity throughout. To use a metaphor, it's like watching four orange fireworks
that share the same uh spherical shapes with only subtle variations.
In contrast, the AI generated music while also having clear motives in each
section and a relatively consistent style. The motives are however highly
independent and the transitions between the segments lack a unified creative intent,
structure, or repetition. It's as if you're seeing four blue
fireworks each with a complete different form. One uh starts uh as a star and
then a line shape and a sphere and so on. So this gap in cohesion becomes even
more apparent in longer compositions and it is uh more obvious when generating
music without uh such coherent lyrics as prompts.
To address this issue of u missing music motive structure in AMG, some teams have
attempt uh attempted to include structural markers in the music encodings such as indicating where
phrases and the current bar belongs to uh and whether it is a repetition of a
previous existing uh phrase. This approach has resulted in a more structured output. This is the ICE EV we
have listened to. [Music]
However, its limitation lies in the difficulty of penotating music structures in the symbolic data,
symbolic music data, particularly when trying to satisfy a structure during uh
generation and the generated music sometimes could be too many repetitions, right?
Okay, let's summarize the technical components covered by today's lecture on AMG. Depending on whether the input is
in audio or symbolic form, we transform in uh transform the input into
computationally understandable encodings and condense them into more context
aware embeddings. And then there are many decoders and generating models to select for auto
reggressive music generation. Throughout the generation process, we utilize a
variety of technologies to enforce the desired constraints and controls. For
example, if I want to generate music on uh a children's Pokemon watch based on
their selected mode today, I will choose MIDI as input because it
is most available and small in size. I will choose to use CB encoding because
it is more condensed and and is also um computationally light and also I will
choose mark of model uh because it is fast and also I will will control it by
rules and sentiment parameters and then if you want to create your own
AMG model you can you know follow the similar uh manners to choose
your uh different modules and connect together
and these are the models uh generating this EV music. So which one do you pick?
Pokemon trainers.
Okay. So the existing techn technological framework already allows for the creation of enjoyable music.
However, a crucial missing piece of the puzzle is the capability to tailor
generated music to individual and specific human activities. Now I will
explain how we are approaching and solving this challenge.
We study how to generate music that a user prefers and promotes beneficial
human activities. Our music generation study starts from rhythmic auditory
stimulations uh short for is an effective intervention to
stabilize the gate of Parkinson's disease patients. RS imposes outer rhythms to help
patients better synchronize their inner clock when walking.
In the past, are conducted under metronomes. Sounds like this.
It is quite tedious, right? Making the patients easy to give up RS
rehabilitations halfway. But after music comes in, RS becomes
more enjoyable.
So how to generate such useful music? Specifically the four aspects of
requirements for qualified music for includes the music should have four full
four time signature and the strong bits as domain specific requirement of
also the music should be in line with certain music theory rules as musicality
requirement to make them sound pleasant. Further it should match personal music
preference and also this uh generation process should keep improving this
satisfaction. Therefore existing songs in music library are not feasible to use for
because very few of them have consistent tempo uh consistent time signature or
drum beats bringing dangers to the patients. And what's worse due to the copyright
issues the music in libraries cannot be used in uh commercial products for
so this work aims to provide usable and cost effective with automatic music
generation motivate uh the patients to do the rehabilitation more often.
Our research involves uh extracting uh individuals music preference from a
user's history of listening to music, their user profile, their interactions
within a music community and also the real-time user states the the body
states. I mean we also focus on integrating this extracted music preference along with domain specific
requirements uh for like into AMG to personalize the
creation of enjoyable and useful music. If you you are interested you can visit
our labs website for more information.
In future work, the primary focus of research will likely center on building
more powerful models uh capable of capturing longer dependencies and
creating more structured representations of music data. Additionally, we can
explore methods for controlling the models and enhancing their interpretability,
ensuring stable and controllable AMG quality. Of course, there is ample room
for further improvements, ultimately aiming to uh elevate the performance of
AMG to the level of human composers or even surpassing them.
Certainly there are also several related topics that fall within the broader
scope of AMG uh including automatic lyrics writing accompiment and
orchestration uh in songwriting uh modeling music performance automatic uh music analysis
and computer understanding of music among others. I believe you will have
chances to explore some of them in the final project of this course. So if
you're interested in this area, uh keep focusing on on the relevant updates of
the uh music information retrieval uh MIR community.
I have also compiled of uh a list of practical open-source data set which
include audio, medi and classification uh young labels, lyrics and more. So
everyone is welcome to use these resources for training your own music generation models. However, it's
important to note that the availability of data for music uh music related task
is currently limited compared to like text and uh image tasks. This scarcity
is a significant uh bottleneck in the field of AMG. We hope that in the
future more high quality data will drive uh the research in this field.
Thank you for attending today's lecture. Uh
if you have any further questions, please feel free to contact me. Thank you.